# ChatSheetPT

## Phase 0: Setting up the project

### Install Dependencies (requirements.txt)

All Dependencies can be installed using the `requirements.txt` file.

In [1]:
%pip install -r backend/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from pathlib import Path

def get_env_variable(var_name: str) -> str:
    value = os.getenv(var_name)
    if value is None:
        raise ValueError(f"Environment variable '{var_name}' not found.")
    return value

In [70]:
from dotenv import load_dotenv

load_dotenv("backend/.env")

azure_api_key = get_env_variable("AZURE_BOSCH_API_KEY")
azure_eval_api_key = get_env_variable("AZURE_OPENAI_API_KEY")
azure_endpoint = get_env_variable("AZURE_BOSCH_ENDPOINT")
azure_embeddings_endpoint = get_env_variable("AZURE_BOSCH_EMBEDDINGS_ENDPOINT")
azure_eval_endpoint = get_env_variable("AZURE_OPENAI_EVAL_ENDPOINT")

## Phase 1: Extraction using Docling

Extraction starts with the cleaning and extraction of raw data from the PDF, to converted it into structured formats.

In [9]:
file_path = "./backend/assets/pdfs/MTS2916A.pdf"

In [10]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2
pipeline_options.do_picture_classification = True
pipeline_options.do_formula_enrichment = True

converter = DocumentConverter(format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options)
        })

/Users/maxtyrchan/Developer/hdm/DesignSpecs-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
try:
    result = converter.convert(file_path)
    doc = result.document
    if not doc:
        raise ValueError("Document conversion produced no output")
except Exception as e:
    raise Exception(f"Error converting document: {str(e)}")

/Users/maxtyrchan/Developer/hdm/DesignSpecs-RAG/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/maxtyrchan/Developer/hdm/DesignSpecs-RAG/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/maxtyrchan/Developer/hdm/DesignSpecs-RAG/.venv/lib/python3.13/site-packages/docling/pipeline/standard_pdf_pipeline.py:262: RuntimeWarning: Mean of empty slice
  np.nanmean(
/Users/maxtyrchan/Developer/hdm/DesignSpecs-RAG/.venv/lib/python3.13/site-packages/docling/pipeline/standard_pdf_pipeline.py:267: RuntimeWarning: Mean of empty slice
  np.nanmean(


## Phase 2: Chunking and Contextualization

In [13]:
from docling.chunking import HybridChunker
import tiktoken
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer,
    ChunkingSerializerProvider,
)
from docling_core.types.doc.labels import DocItemLabel
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer
from docling_core.transforms.serializer.markdown import MarkdownParams

from docling_core.transforms.serializer.base import (
    BaseDocSerializer,
    SerializationResult,
)
from docling_core.transforms.serializer.common import create_ser_result
from docling_core.transforms.serializer.markdown import MarkdownPictureSerializer
from docling_core.types.doc.document import (
    PictureClassificationData,
    PictureDescriptionData,
    PictureItem,
    PictureMoleculeData,
)


class SerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(
            doc=doc,
            # configuring a different table serializer
            table_serializer=MarkdownTableSerializer(),
            # Leave out images eventually because we already have them in the image_summaries
            # params=MarkdownParams(
            # image_placeholder="<!-- image -->",
            # ),
        )


tokenizer = OpenAITokenizer(
    tokenizer=tiktoken.encoding_for_model("gpt-4o"),
    max_tokens=128 * 1024,  # context window length required for OpenAI tokenizers
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,  # optional, defaults to True
    serializer_provider=SerializerProvider(),
)

chunks = list(chunker.chunk(dl_doc=doc))

In [14]:
partitioned_data = {
    "texts": [],
    "texts_summaries": [],
    "images": [],
    "images_summaries": [],
    "tables": [],
    "tables_summaries": []
}

### Extract Text and Tables

In [15]:
for chunk in chunks:
    ser_txt = chunker.contextualize(chunk=chunk)
    for it in chunk.meta.doc_items:
        if it.label == DocItemLabel.TABLE:
            partitioned_data["tables"].append(chunk)
            partitioned_data["tables_summaries"].append(ser_txt)
        if it.label == DocItemLabel.TEXT:
            # chunk.text to only include the text | chunk for text and metadata
            partitioned_data["texts"].append(chunk)
            partitioned_data["texts_summaries"].append(ser_txt)

### Extract Images

In [16]:
import base64

for picture in doc.pictures:
    img = picture.get_image(doc)
    if any(annotation.kind == "description" for annotation in picture.annotations):
        try:
            b64 = picture._image_to_base64(img)
            decoded_image = base64.b64decode(b64, validate=True)
            if decoded_image:
                partitioned_data["images"].append(b64)
        except Exception as e:
            print(f"Error decoding image: {e}")
        for annotation in picture.annotations:
            if annotation.kind == "description":
                partitioned_data["images_summaries"].append(
                    annotation.text)
    else:
        pass

In [16]:
print(partitioned_data["images_summaries"][3])

The image depicts a circuit diagram, specifically focusing on a voltage divider. The diagram includes several components:

1. **Power Bridge**:
   - **VREF**: The voltage divider is connected to the VREF.
   - **VOUT**: The voltage divider is connected to the VOUT.

2. **One-Shot**:
   - **VREF**: The voltage divider is connected to the VREF.
   - **VOUT**: The voltage divider is connected to the VOUT.

3. **Comp1**:
   - **VREF**: Comp1 is connected to the VREF.
   - **VOUT**: Comp1 is connected to the VOUT.

4. **Comp2**:
   - **VREF**: Comp2 is connected to the VREF.
   - **VOUT**: Comp2 is connected to the VOUT.

5. **Comp3**:
   - **VREF**: Comp3 is connected to the VREF


## Phase 3: Vectorization

Removes the old databases!

In [ ]:
#!rm -rf backend/db

Wrapper class for AzureOpenAIEmbeddings for ChromaDB compatibility

In [ ]:
class Embedding:
    def __init__(self, embedding_model):
        self.embeddings = embedding_model
        self.text_item = "text_item"

    def name(self) -> str:
        return "AzureOpenAIEmbeddings"

    def __call__(self, input):
        return self.embeddings.embed_documents(input)

    def embed_query(self, text: str):
        return self.embeddings.embed_query(text)

    def embed_documents(self, texts):
        return self.embeddings.embed_documents(texts)

Initializes the embedding model and creates databases.

In [19]:
from pathlib import Path
from langchain_chroma import Chroma
import chromadb
from langchain.storage import LocalFileStore
from langchain.schema.document import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
import os
from datetime import datetime
import shutil

embeddings = Embedding(AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    azure_endpoint=azure_embeddings_endpoint,
    openai_api_key=azure_api_key,
    api_version="2024-02-01",
))

# db_path = Path("./db/chroma")
db_path = Path("old_notebooks/db/chroma")
if db_path.exists():
    shutil.rmtree(db_path)

# Initialize Chroma with persistent storage and proper settings
# chroma_client = chromadb.PersistentClient(
#     path=str(db_path),
#     settings=chromadb.Settings(
#         allow_reset=True,
#         is_persistent=True,
#         anonymized_telemetry=False
#     )
# )

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="DesignSpecsRAG",
    embedding_function=embeddings,
    metadata={"created": str(datetime.now())}
)

# Initialize vector store
vectorstore = Chroma(
    collection_name="DesignSpecsRAG",
    embedding_function=embeddings,
    client=chroma_client
)

# The storage layer for the parent documents
file_store = LocalFileStore("./db/docs")

id_key = "doc_id"

# Initialize retriever with fixed bytes handling
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=file_store,
    # Retrieve more documents to include tables and images
    search_kwargs={"k": 10}
)

Loads Documents with links to the original Documents into the Vectorstore and Document Store

In [20]:
import json
import uuid

# Add texts            
doc_ids = [str(uuid.uuid4()) for _ in partitioned_data["texts"]]
summary_texts = [Document(page_content=summary, metadata={retriever.id_key: doc_ids[i], "content_type": "text"}) for i, summary in enumerate(partitioned_data["texts_summaries"])]
#"text_meta": partitioned_data["texts"][i].meta}
texts = [json.dumps(text.export_json_dict()).encode('utf-8') for text in partitioned_data["texts"]]

retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in partitioned_data["tables"]]
summary_tables = [Document(page_content=summary, metadata={retriever.id_key: table_ids[i],"content_type": "table"}) for i, summary in enumerate(partitioned_data["tables_summaries"])]
#,"table_meta": partitioned_data["tables"][i].meta
tables = [json.dumps(table.export_json_dict()).encode('utf-8') for table in partitioned_data["tables"]]

retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(doc_ids, tables)))

# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in partitioned_data["images"]]
summary_img = [Document(page_content=summary, metadata={retriever.id_key: img_ids[i],"content_type": "image", }) for i, summary in enumerate(partitioned_data["images_summaries"])]
images = [img.encode('utf-8') for img in partitioned_data["images"]]

retriever.vectorstore.add_documents(summary_img)
retriever.docstore.mset(list(zip(img_ids, images)))

## Phase 4: Retrival Augmented Generation (RAG)

Split content into images, tables, and regular text based on metadata

In [41]:
from base64 import b64decode
import json


def parse_docs(docs):
    """Split base64-encoded images and everything else (texts/tables)"""
    b64_images = []
    texts = []

    for doc_id, doc_content in enumerate(docs):  # Unpack the tuple (uuid, content)
        
        if hasattr(doc_content, 'page_content'):
            # It's a Document object
            content_str = doc_content.page_content
        elif isinstance(doc_content, bytes):
            # It's raw bytes from docstore
            content_str = doc_content.decode('utf-8')
        elif hasattr(doc_content, 'export_json_dict'):
            # It's a DoclingDocument object
            content_str = json.dumps(doc_content.export_json_dict())
        elif isinstance(doc_content, str):
            # It's a string
            content_str = doc_content
        else:
            print(f"DEBUG: Unexpected doc type: {type(doc_content)}")
            content_str = str(doc_content)

        if isinstance(content_str, str) and content_str.startswith("b'") and content_str.endswith("'"):
               # Remove the b' prefix and ' suffix to get the actual content
            try:
                content_str = content_str[2:-1]
                # Handle escaped characters
                content_str = content_str.encode().decode('unicode_escape')
            except Exception as e:
                print(f"Failed to extract from byte string representation: {e}")
        
        # Check if it's a base64-encoded image
        try:
            # Try to decode as base64 first - if this works, it's an image
            b64decode(content_str, validate=True)
            b64_images.append(content_str)
        except:
            # If base64 decode fails, it's a JSON-encoded chunk (text or table)
            texts.append(content_str)
    return {"images": b64_images, "texts": texts}

Constructs the prompt for the RAG system by including images, tables, and text as context.

In [22]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage
import json


def build_prompt(kwargs):
    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = ""

    # Process texts (they are now JSON strings or plain text)
    if len(docs_by_type["texts"]) > 0:
        for text_element in docs_by_type["texts"]:
            try:
                # Try to parse as JSON (chunk data)
                doc_json = json.loads(text_element)
                content_text = doc_json.get('text', '')
                context_text += content_text + "\n"
            except json.JSONDecodeError:
                # If it's not JSON, treat as plain text
                context_text += str(text_element) + "\n"

    # Construct prompt with context (including images)
    prompt_template = f"""
        Answer the question based only on the following context, which includes text, tables, and images (if present).
        
        Context: {context_text}
        
        Question: {user_question}
        """

    prompt_content = [{"type": "text", "text": prompt_template}]

    # Add images
    if len(docs_by_type["images"]) > 0:
        for image in docs_by_type["images"]:
            prompt_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            })

    return ChatPromptTemplate.from_messages([HumanMessage(content=prompt_content)])

Initializes the llm and reranker for the RAG system.

In [29]:
from langchain_litellm import ChatLiteLLM
from typing import List, Optional
from langchain_community.document_compressors import FlashrankRerank
from langchain.callbacks.tracers import LangChainTracer

llm = ChatLiteLLM(
    api_key=azure_api_key,
    api_base=azure_endpoint,
    temperature=0,
    model="azure/gpt-4o",
)

tracer = LangChainTracer(project_name=get_env_variable("LANGSMITH_PROJECT"))

compressor = FlashrankRerank()

compressor.model = "rank-T5-flan"

def rerank_documents(compressor: FlashrankRerank, query: str, documents: List[Document], top_n: Optional[int] = None) -> List[Document]:
    if not documents:
        return []

    try:
        compressed_docs = compressor.compress_documents(
            documents=documents,
            query=query
        )

        if top_n is not None:
            return compressed_docs[:top_n]
        return compressed_docs

    except Exception as e:
        # Fallback: return original documents
        if top_n is not None:
            return documents[:top_n]
        return documents

Option 1: Without Reranking

In [30]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from typing import Dict, Any


async def answer_question(question: str) -> Dict[str, Any]:
    """
    Args:
    question: The question to answer
    Returns:
    Dictionary containing the answer and relevant context
    """
    try:
        # Response with sources
        chain_with_sources = {
            "context": retriever | RunnableLambda(parse_docs),
            "question": RunnablePassthrough(),
        } | RunnablePassthrough().assign(
            response=(
                RunnableLambda(build_prompt)
                | llm
                | StrOutputParser()
            )
        )

        response = await chain_with_sources.ainvoke(input=question, config={"callbacks": [tracer]})

        return {
            "answer": response['response'],
            "context": response['context']
        }

    except Exception as e:
        print(f"Error during chain invocation: {e}")
        return {
            "answer": "I apologize, but I'm currently experiencing technical difficulties. Both primary and fallback services are unavailable. Please try again later.",
            "context": {"images": [], "tables": [], "texts": []}
        }

Option 2: With Reranking

In [39]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from typing import Dict, Any


async def answer_question_with_reranking(question: str) -> Dict[str, Any]:
    """
    Args:
    question: The question to answer
    Returns:
    Dictionary containing the answer and relevant context
    """
    try:
        docs = await retriever._aget_relevant_documents(
            query=question, run_manager=tracer)

        reranked_docs = rerank_documents(compressor= compressor, query=question, documents=docs, top_n=10)

        # Convert the list to a Runnable that returns the list
        docs_runnable = RunnableLambda(lambda _: reranked_docs)

        # Response with sources
        chain_with_sources = {
            "context": docs_runnable | RunnableLambda(parse_docs),
            "question": RunnablePassthrough(),
        } | RunnablePassthrough().assign(
            response=(
                RunnableLambda(build_prompt)
                | llm
                | StrOutputParser()
            )
        )

        response = await chain_with_sources.ainvoke(input=question, config={"callbacks": [tracer]})

        return {
            "answer": response['response'],
            "context": response['context']
        }

    except Exception as e:
        print(f"Error during chain invocation: {e}")
        return {
            "answer": "I apologize, but I'm currently experiencing technical difficulties. Both primary and fallback services are unavailable. Please try again later.",
            "context": {"images": [], "tables": [], "texts": []}
        }

Functions to display the context

In [32]:
import base64
from IPython.display import Image, display


def display_base64_image(base64_code):
    # Decode the base64 string to binary
    image_data = base64.b64decode(base64_code)
    # Display the image
    display(Image(data=image_data))

In [33]:
import json


def display_context_details(response):
    """Display separated text and table content from the response"""

    texts = []
    tables = []

    for text_element in response['context']['texts']:
        try:
            doc_json = json.loads(text_element)
            content_text = doc_json.get('text', '')
            
            # Get metadata
            meta = doc_json.get('meta', {})
            doc_items = meta.get('doc_items', [])
            origin = meta.get('origin', {})

            # Check if it's a table
            has_table = any(item.get('label') == 'table' for item in doc_items)

            # Extract page info
            page_no = 'Unknown'
            if doc_items and 'prov' in doc_items[0]:
                page_no = doc_items[0]['prov'][0].get('page_no', 'Unknown')

            item_data = {
                'content': content_text,
                'page': page_no,
                'filename': origin.get('filename', 'Unknown')
            }

            if has_table:
                tables.append(item_data)
            else:
                texts.append(item_data)

        except json.JSONDecodeError:
            texts.append({
                'content': str(text_element),
                'page': 'Unknown',
                'filename': 'Unknown'
            })

    return {
        'texts': texts,
        'tables': tables,
        'images': response['context']['images']
    }

Testing the RAG pipeline with a question

In [ ]:
reranked_response = await answer_question_with_reranking("What is the Power Bridge Operation of the MTS2916A?")

print("Response:", reranked_response['answer'])

INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
08:55:05 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4o; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o; provider = azure
INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
08:55:09 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20
08:55:09 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20


Response: The power bridge operation of the MTS2916A involves driving the two windings of a bipolar stepper motor using an H-type bridge. Each motor winding is controlled by an H-bridge consisting of two N-transistors and two P-transistors. This configuration allows current to flow in both directions through the winding, depending on the value of the **PHASE** signal. 

The H-bridge can be set in five configurations based on the digital inputs **PHASE**, **I0**, and **I1**, as well as the current sensed. These configurations determine the direction and magnitude of the current through the motor windings, enabling precise control of the stepper motor's operation.


09:07:51 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 401 PermissionDenied"



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



09:07:52 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 401 PermissionDenied"
09:07:52 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure


...

INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 401 PermissionDenied"
09:29:53 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 401 PermissionDenied"



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



09:39:51 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
09:39:53 - LiteLLM:INFO: utils.py:1215 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
09:39:53 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4.1-2025-04-14
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4.1-2025-04-14
09:39:55 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4.1; provider = azure
INFO:httpx:HTTP Request: POST https://mt103-mamfd3cj-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-vers

In [43]:
response = await answer_question("What is the Power Bridge Operation of the MTS2916A?")

print("Response:", response['answer'])

INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
08:55:27 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4o; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o; provider = azure
INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
08:55:30 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20
08:55:30 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20


Response: The power bridge operation of the MTS2916A involves driving the two windings of a bipolar stepper motor using an H-type bridge. Each motor winding is controlled by an H-bridge consisting of two N-type and two P-type transistors. This configuration allows current to flow in both directions through the winding, depending on the value of the **PHASE** signal. 

The H-bridge can be set in five configurations based on the digital inputs **PHASE**, **I0**, and **I1**, as well as the current sensed. These configurations determine the direction and magnitude of the current through the motor windings, enabling precise control of the stepper motor's operation.


Display the context

In [46]:
context = display_context_details(response)

In [47]:
for text in context['texts']:
    print(f"Text: {text['content']}, Page: {text['page']}, Filename: {text['filename']}")

Text: NOTES:, Page: 16, Filename: MTS2916A.pdf
Text: NOTES:, Page: 18, Filename: MTS2916A.pdf
Text: The circuit is designed to drive the two windings of a bipolar stepper motor, and can be divided into two identical channels (channel 1 and channel 2) and protection circuitry  for  overtemperature  and  undervoltage.  The functionality  of  a  channel  and  protection  circuitry  is presented in the following sections.
Each  motor  winding  is  driven  by  an  H-type  bridge consisting of two N- and two P-transistors that allow the current to flow in both winding directions, depending on the value of the PHASE  signal  (Table 3-1). The H-bridge  can  be  set  in  five  configurations  that  are related to the digital inputs PHASE, I0 and I1, and to the current  sensed.  These  configurations  are  shown  in Table 3-2.
FIGURE 3-1: Power Bridge Control (PHASE = H/Forward).

bar chart

The image shows a diagram with a series of arrows pointing from one arrow to another. The arrows are labe

In [48]:
for table in context['tables']:
    print(f"Table: {table['content']}, Page: {table['page']}, Filename: {table['filename']}")

In [ ]:
display_base64_image(context['images'][0])

## Phase 5: Evaluation

Initializes the evaluation system for the RAG pipeline.

In [71]:
from langsmith import Client
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT, CONCISENESS_PROMPT, HALLUCINATION_PROMPT, RAG_GROUNDEDNESS_PROMPT, RAG_HELPFULNESS_PROMPT, RAG_RETRIEVAL_RELEVANCE_PROMPT
from langchain_litellm import ChatLiteLLM

# Define the input and reference output pairs that you'll use to evaluate your app
client = Client()

eval_llm = ChatLiteLLM(
    model="azure/gpt-4.1",
    temperature=0,
    api_base=azure_eval_endpoint,
    api_key=azure_eval_api_key
)

# Check if dataset already exists
existing_datasets = client.list_datasets()
# dataset_name = "small_eval_dataset"
dataset_name = "eval_dataset"

for ds in existing_datasets:
    if ds.name == dataset_name:
        dataset = ds
        break
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name, description="A  dataset with simple questions."
    )

    # Export examples for use in API - only add when creating new dataset
    examples = [
        {
            "inputs": {"question": "What are the selectable output current limits?"},
            "outputs": {"answer": "The selectable output current limits are 0%, 33%, 67%, or 100% of the maximum output current."},
        },
        {
            "inputs": {"question": "Which stepping operations are possible with the PWM?"},
            "outputs": {"answer": "Full, half, and micro-stepping operations are possible with the PWM current control and logic inputs."},
        },
        {
            "inputs": {"question": "Is a special power-up sequencing required?"},
            "outputs": {"answer": "No special power-up sequencing is required."},
        },
        {
            "inputs": {"question": "What does a high and what does a low logic signal level cause?"},
            "outputs": {"answer": "A HIGH logic signal level causes load current to flow from OUTxA to OUTxB. A LOW logic level causes load current to flow from OUTxB to OUTxA."},
        },
        {
            "inputs": {"question": "What does a thermal protection circuitry do?"},
            "outputs": {"answer": "A thermal protection circuitry turns off all drivers when the junction temperature exceeds a safe operating limit of +170°C (typical)."},
        },
        {
            "inputs": {"question": "When are the power bridge and all outputs are disabled?"},
            "outputs": {"answer": "The power bridge and all outputs are disabled if VLOGIC is smaller than 4V."},
        },
        {
            "inputs": {"question": "What do the typical PCB layout guidelines include?"},
            "outputs": {"answer": "Separate power ground planes, supply decoupling capacitors close to the IC, short connections and use of maximized copper areas to improve thermal dissipation."},
        },
        {
            "inputs": {"question": "What is the MTS2916A DUAL FULL-BRIDGE STEPPER MOTOR DRIVER?"},
            "outputs": {"answer": "The MTS2916A Dual Full-Bridge Stepper Motor Driver Evaluation Board control circuitry is designed to typically operate from a 6V to 12V logic input (internally regulated down to 5V) and a 10V to 30V VLOAD input. VLOAD provides power to the motor windings. Test points are generously distributed throughout the evaluation board. This gives the user easy access and visibility, facilitating a better understanding of the MTS2916A operating details."},
        },
        {
            "inputs": {"question": "Which Power Connections does the MTS2916A use?"},
            "outputs": {"answer": "The MTS2916A Dual Full-Bridge Stepper Motor Driver Evaluation Board uses a combination of terminal blocks, test clips and one DC power jack for power connections. Connections are as follows: a) Motor Output Connections: - J2-1(A3), J2-2(A1), J2-3(B1), J2-4(B3), J2-5(TP21) - TP11(A1), TP12(A3), TP13(B1), TP14(B3). b) VLOAD (Motor Supply Power): - J4-1(PGND), J4-2(VLOAD) - TP20(PGND), TP18(VLOAD). WARNING: Do not connect more than 16V to these motor supply connections while Jumper JP2 is installed. c) VLOGIC: - J1-1(VLOGIC), J1-2(AGND) - TP2(VLOGIC), TP5(AGND)"},
        },
        {
            "inputs": {"question": "What are the steps to power the MTS2916A Dual Full-Bridge Stepper Motor Driver Evaluation Board?"},
            "outputs": {"answer": "Follow these steps to power-up the board: 1. With the supply turned OFF, connect the power to the logic portion of the evaluation board at J1 with the specified voltage (7 VDC to 12 VDC). The logic portion of the evaluation board will typically draw less than 50 mA. 2. If the user’s stepper motor requires a voltage that is compatible with the logic supply voltage and the user’s source can handle driving the stepper motor windings, install JP2. DO NOT connect power at J4. If powering up the stepper from an additional supply, DO NOT install JP2 and connect the stepper motor supply to J4. J1 power will still be required for the logic supply. 3. 4. Connect the bipolar stepper windings to J2 per the schematic diagram. Turn ON the power supplies. Power sequencing is not required due to the under- voltage lockout circuitry. 5. Toggle the Mode switch to cycle through the five modes, as indicated by the binary LED count. 6. Press the Run switch once to tell the PIC16F883 to send drive information to the MTS2916A with minimal (1V) VREF . Subsequent Run presses increase VREF by approximately 1V up to 5V maximum. This increases the current regulation threshold. 7. The Hold switch tells the PIC16F883 to command the MTS2916A to hold the motor position. 8. The Direction switch tells the PIC16F883 to command the MTS2916A to change the direction of the motor. 9. The Speed Adjust Potentiometer (R4) varies an analog voltage that is read by the PIC16F883 Analog-to-Digital Converter, and varies the speed accordingly."},
        },
    ]

    # Add the examples to the dataset only when creating new dataset
    client.create_examples(dataset_id=dataset.id, examples=examples)

Define the evaluation function to assess the RAG system's performance.

In [ ]:
def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate correctness of the output."""

    evaluator = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="correctness",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )
    return eval_result


def conciseness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate relevance of the output."""
    evaluator = create_llm_as_judge(
        prompt=CONCISENESS_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="conciseness",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )
    return eval_result


def hallucination_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate relevance of the output."""
    # Extract context from outputs - simple format
    context_text = ""
    if "context" in outputs and isinstance(outputs["context"], dict) and "texts" in outputs["context"]:
        context_text = "\n".join(outputs["context"]["texts"])

    evaluator = create_llm_as_judge(
        prompt=HALLUCINATION_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="hallucination",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs["answer"],
        context=context_text,
        reference_outputs=reference_outputs
    )
    return eval_result


def groundedness_evaluator(outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate relevance of the output."""
    # Extract context from outputs - simple format
    context_text = ""
    if "context" in outputs and isinstance(outputs["context"], dict) and "texts" in outputs["context"]:
        context_text = "\n".join(outputs["context"]["texts"])

    evaluator = create_llm_as_judge(
        prompt=RAG_GROUNDEDNESS_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="groundedness",
    )
    eval_result = evaluator(
        outputs=outputs["answer"],
        context=context_text,
        reference_outputs=reference_outputs
    )
    return eval_result


def relevance_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate relevance of the output."""
    # Extract context from outputs - simple format
    context_text = ""
    if "context" in outputs and isinstance(outputs["context"], dict) and "texts" in outputs["context"]:
        context_text = "\n".join(outputs["context"]["texts"])

    evaluator = create_llm_as_judge(
        prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="relevance",
    )
    eval_result = evaluator(
        inputs=inputs,
        context=context_text,
        reference_outputs=reference_outputs
    )
    return eval_result


def helpfulness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict):
    """Define an LLM as a judge evaluator to evaluate relevance of the output."""
    evaluator = create_llm_as_judge(
        prompt=RAG_HELPFULNESS_PROMPT,
        judge=eval_llm,
        model="gpt-4.1",
        feedback_key="helpfulness",
    )
    eval_result = evaluator(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs
    )
    return eval_result

In [64]:
async def target(inputs: dict) -> dict:
    question = inputs["question"]
    try:
        answer = await answer_question_with_reranking(question=question)

    except Exception as e:
        print(f"Error during question answering: {e}")
        pass

    # Extract just the text content for evaluation (no metadata needed)
    context_texts = []
    if answer and "context" in answer and isinstance(answer["context"], dict) and "texts" in answer["context"]:
        context = display_context_details(answer)
        for text_item in context["texts"]:
            if isinstance(text_item, dict) and "content" in text_item:
                context_texts.append(text_item["content"])
            elif isinstance(text_item, str):
                context_texts.append(text_item)

    return {
        "answer": answer["answer"],
        "context": {
            "texts": context_texts
        }
    }

Function to trigger the evaluation process.

In [ ]:
async def trigger_evaluation() -> Dict[str, Any]:
    """Trigger a new evaluation run"""
    experiment_results = await client.aevaluate(
        target,
        # data="small_eval_dataset",
        data="eval_dataset",
        evaluators=[
            correctness_evaluator,
            conciseness_evaluator,
            hallucination_evaluator,
            groundedness_evaluator,
            relevance_evaluator,
            helpfulness_evaluator
        ],
        experiment_prefix="rag-eval",
        max_concurrency=1,
    )

    return experiment_results

In [72]:
eval_results = await trigger_evaluation()

View the evaluation results for experiment: 'rag-eval-b1d882ec' at:
https://eu.smith.langchain.com/o/f02db528-b643-497a-ae69-37425f8940c0/datasets/04ac1acb-00e6-430c-a9a8-79ff29515416/compare?selectedSessions=ac5cbc95-ed7c-426c-99a1-bc225bed0c10




0it [00:00, ?it/s]INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
09:39:39 - LiteLLM:INFO: utils.py:3043 - 
LiteLLM completion() model= gpt-4o; provider = azure
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o; provider = azure
INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2025-02-01-preview "HTTP/1.1 200 OK"
09:39:49 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20
09:39:49 - LiteLLM:INFO: cost_calculator.py:655 - selected model name for cost calculation: azure/gpt-4o-2024-11-20
INFO:LiteLLM:selected model name for cost calculation: azure/gpt-4o-2024-11-20
1it [00:23, 23.11s/it]INFO:httpx:HTTP Request: POST https://dr-open-ai.openai.azure.com/openai/deployments/text-

In [ ]:
print(eval_results)

{'metrics': [], 'pairs': [{'id': 'e1e1e591-e2b0-4d08-90af-88f294790edf', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': 'c0c99eb5-d56c-468b-ba35-939c4f004a17', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': '60cd9511-3ba3-4d67-879c-5345a43cc145', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': '886dee19-4df2-4f9c-ad1e-5604a35aff5d', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': '0e53f34a-5a4f-42db-a02a-490dea665ce2', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': '3416a402-af9e-41da-92a8-797bfe4e2198', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': 'cbc89edf-c8ad-4073-a199-88ae1118f7be', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': '6dc2a008-a07f-45e4-8fc3-0241678342ac', 'query': '', 'response': '', 'timestamp': None, 'metrics': []}, {'id': 'b60c8b8b-9312-492d-aa5d-04d09062436f', 'query': '', 'response': '', 'timestamp